# 3 · Prototype case studies

The prototype (`prototype/app.py`) takes a location and a start month, downloads **every raw Sentinel-1 scene** of
12 months (plus the same 3 months one year later), and predicts one map per month with the final setting: an
ensemble of 4 models (E3, E1, E5, E2b), month-mean probabilities and 3-month smoothing.

**Built-up change** compares the first 3 months with the **same 3 calendar months one year later** (same season, so
monsoon or dry-season effects cannot read as construction). Three rules: *argmax* (built-up in most end months, in
none of the start months), *moderate* and *confident* (built-up probability ≥ 0.5 at the end and below 0.35 / 0.2 in
every start month). **Moderate is the headline rule**: in the gallery it reported almost nothing in stable places
and tens of hectares where both references see growth, while confident missed obvious growth and argmax reported
change in stable places. The verdict compares the new built-up with the model's error level measured on 130 unseen test
places. Every place is checked against two products never used in training: **Open Buildings 2.5D Temporal** and
**ESRI 10 m land cover**.

Places computed with `prototype/case_studies.py` (July 2022 – June 2023 + July–Sept 2023, 5.12 km × 5.12 km); several
were chosen *because* they are hard for radar.

### How the prototype works, step by step

1. **Area.** The clicked point becomes the centre of a 512 × 512 pixel square (5.12 km) in the local UTM projection,
   the same 10 m grid recipe as the training data.
2. **Download.** Every Sentinel-1 scene of the 12 months + 3 more months is downloaded with exactly the training
   recipe (`data/s1_composite.py` is shared by training and prototype, so the input looks identical).
3. **Predict.** Every scene goes through the 4 models (ensemble); class probabilities are averaged over models and
   over the scenes of each month, then over neighbouring months (smooth3); argmax gives the monthly maps.
4. **Statistics.** % of area per class per month, month-to-month change and total change in percentage points.
5. **Built-up change.** First 3 months vs the same 3 months a year later, with the moderate rule (and the other two
   as bounds).
6. **Verdict.** New built-up compared with the error level (what the model reports as false new built-up on test
   places of the same size): ≥ 3× = clear growth, 1.5–3× = some growth, below = no change detectable.
7. **Independent checks.** ESRI 10 m land cover (two years) and Open Buildings Temporal (yearly building presence),
   neither used in training.

**Why the same-season comparison?** In a first version, built-up change compared the first 3 with the last 3 of the
12 months, i.e. different seasons. In Dhaka (monsoon floods) or Nairobi (rainy vs dry season) the radar signal changes
with the season, and the argmax rule reported hundreds of hectares of "new built-up" that were really wet vs dry
soil. Comparing July–September with July–September one year later removes the season from the comparison.

**Why a gallery?** The places were computed in advance (`case_studies.py`) so the app can show them instantly in
the screen recording; a live run of a new place takes 2–5 minutes.

In [ ]:
import json, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent                  # the s1-lulc folder
sys.path[:0] = [str(ROOT / "data"), str(ROOT / "train"), str(ROOT / "prototype")]
DATA, PREP, RUNS = ROOT / "data" / "tiles_v0", ROOT / "data" / "tiles_v0" / "prepared", ROOT / "train" / "runs"
from classes import CLASS_NAMES, DISPLAY_NAMES, CLASS_COLOURS, N_CLASSES, BUILT
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project folder:", ROOT)

In [ ]:
import gzip, pickle, base64
from IPython.display import HTML
import pipeline as P
CASES = sorted((ROOT / "prototype" / "case_studies").glob("*.pkl.gz"))
cases = []
for f in CASES:
    with gzip.open(f, "rb") as fh:
        cases.append(pickle.load(fh))
from case_studies import CASES as CASE_LIST, FINDINGS
order = [c[0] for c in CASE_LIST]
cases.sort(key=lambda r: order.index(r["case"]["id"]) if r["case"]["id"] in order else 99)
RULE, MODE = P.PROTOTYPE["change_rule"], P.PROTOTYPE["change_mode"]
print(len(cases), "places:", ", ".join(c["case"]["title"] for c in cases))
print(f"prototype setting: models {'+'.join(P.PROTOTYPE['runs'])}, 3-month smoothing {P.PROTOTYPE['smooth3']}, "
      f"change rule {RULE}, comparison {MODE.replace('_', ' ')}")

In [ ]:
def change(res, mode=MODE, rule=RULE):
    pr, pb, mo, fi, la, used = P.change_inputs(res, mode)
    return P.built_change(pr, mo, pbuilt=pb, rule=rule, first=fi, last=la), used

def built_refs(res, bc):
    refs = []
    if bc is None:
        return refs
    if res.get("obt"):
        o = res["obt"]
        refs.append(P.reference_change(f"Open Buildings Temporal ({o['date_a'][:4]} vs {o['date_b'][:4]})",
                                       o["built_a"], o["built_b"], o["valid"], bc))
    if res.get("esri_ab"):
        y0, y1, ea, eb = res["esri_ab"]
        refs.append(P.reference_change(f"ESRI land cover ({y0} vs {y1})", ea == P.BUILT, eb == P.BUILT,
                                       (ea >= 0) & (eb >= 0), bc))
    return refs

RUN_DIRS = [ROOT / "train" / "runs" / r for r in P.PROTOTYPE["runs"]]
def floor_of(res, rule=RULE):
    """Error level of this rule, measured on the test places (computed now from train/runs if available,
    else the value stored when the place was computed)."""
    if all(d.exists() for d in RUN_DIRS):
        return P.noise_floor(RUN_DIRS, rule, P.PROTOTYPE["tta"], smooth3=P.PROTOTYPE["smooth3"])
    c = res["case"]
    return c.get("floors", {}).get(rule, c.get("floor", P.MIN_FLOOR_HA_KM2))

summary = []
for res in cases:
    c = res["case"]; row = {"place": c["title"], "type": c["kind"], "scenes": int(res["n"].sum())}
    bc, used = change(res)
    if bc is None:
        row["verdict"] = "not enough Sentinel-1 data"; summary.append(row); continue
    v, why, exp = P.change_verdict(bc, floor_of(res))
    row.update({"compares": used.replace("_", " "), "built-up start %": bc["before_pct"], "end %": bc["after_pct"],
                f"new ha ({RULE})": bc["gain_ha"], "error level ha": exp, "verdict": v})
    for rule in ("argmax", "moderate"):
        b2, _ = change(res, rule=rule)
        row[f"new ha ({rule})"] = b2["gain_ha"] if b2 else np.nan
    for r in built_refs(res, bc):
        key = "OBT" if r["source"].startswith("Open") else "ESRI"
        row[f"{key} new ha"] = r["new built-up (ha)"]
    summary.append(row)
display(pd.DataFrame(summary).set_index("place").round(1))

### How to read the summary table

* **scenes**: radar scenes used (more = steadier maps; Europe ~150, Africa ~30 per year);
* **compares**: same season (normal) or within year (fallback when the extra months are missing);
* **built-up start / end %**: share of built-up at the start and end period;
* **new ha (moderate / argmax / confident)**: new built-up area with each rule; moderate is the headline,
  argmax the upper bound, confident the strict lower bound;
* **error level ha**: false new built-up expected here from the test-set error rate;
* **verdict**: new built-up compared with the error level;
* **OBT / ESRI new ha**: new built-up according to the two independent products.

**Careful when comparing with the references.** They disagree with each other too (e.g. Dhaka: ESRI 286 ha vs Open
Buildings 25 ha; Kasoa: 0.1 vs 195 ha). ESRI is a 10 m land-cover map with its own yearly noise; Open Buildings
Temporal detects buildings from higher-resolution images but only in some regions. Agreement with *either* is
evidence, not proof.

## Each place in detail

For every place: the 12 monthly radar images, maps and built-up probability (how the whole area changes through the
year), an animation of the monthly maps, the built-up change map and the class shares per month.

In [ ]:
def month_strip(res):
    months, n = res["months"], res["n"]
    pb = res.get("pbuilt")
    rows = 3 if pb is not None else 2
    fig, ax = plt.subplots(rows, 12, figsize=(24, 2.2 * rows + 0.4), squeeze=False)
    for m in range(12):
        if res.get("s1") is not None:
            vv = res["s1"][m, 0].astype(float) / 100; vv[res["s1"][m, 0] == P.S1_NODATA] = np.nan
            ax[0, m].imshow(vv, cmap="gray", vmin=-25, vmax=0)
        ax[1, m].imshow(P.colourise(res["pred"][m]), interpolation="nearest")
        if pb is not None:
            b = pb[m].astype(float); b = b / 255 if b.max() > 1.5 else b
            ax[2, m].imshow(np.where(res["pred"][m] >= 0, b, np.nan), cmap="magma", vmin=0, vmax=1)
        ax[0, m].set_title(f"{months[m]}  ({int(n[m])} scenes)", fontsize=8)
    for a, t in zip(ax[:, 0], ["radar VV", "map", "P(built-up)"]):
        a.set_ylabel(t, fontsize=9)
    for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()

for res in cases:
    c = res["case"]; months = res["months"]
    bc, used = change(res)
    found = FINDINGS.get(c["id"], "")
    display(Markdown(f"### {c['title']}  ·  *{c['kind']}*\n\n**What to look for:** {c['note']}"
                     + (f"\n\n**What we found:** {found}" if found else "")))
    if bc is None:
        display(Markdown("**No result:** too few months with Sentinel-1 data here.")); continue
    v, why, _ = P.change_verdict(bc, floor_of(res))
    display(Markdown(f"**Verdict ({RULE} rule, {used.replace('_', ' ')}): {v}.** {why}"))
    month_strip(res)
    gif = P.animation_gif(res["pred"], months)
    display(HTML(f'<img src="data:image/gif;base64,{base64.b64encode(gif).decode()}" width="360"> '
                 f'<span style="font-size:12px">monthly maps as an animation</span>'))
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.5]})
    ax[0].imshow(P.colourise(P.year_majority(res["pred"]))); ax[0].set_title("year majority (our model)")
    ax[1].imshow(P.change_rgb(bc)); ax[1].set_title(f"built-up {bc['first'][0]}… vs …{bc['last'][-1]}\nmagenta new, cyan lost", fontsize=10)
    pct = P.class_percentages(res["pred"], months)
    for k, col in enumerate(pct.columns):
        ax[2].plot(range(12), pct[col], color=CLASS_COLOURS[k], label=DISPLAY_NAMES[k], marker="o", ms=3)
    ax[2].set_xticks(range(12), [m[2:] for m in months], rotation=45, fontsize=8); ax[2].set_ylabel("% of area")
    ax[2].legend(fontsize=7, ncol=2, frameon=False); ax[2].set_title("class % per month")
    for a in ax[:2]: a.axis("off")
    plt.tight_layout(); plt.show()
    display(P.built_change_table(bc, built_refs(res, bc)).round(2))

### How to read each place

1. **Verdict** with the explanation (how many hectares, how many times the error level).
2. **12-month strip**: radar VV (the best scene of each month, with the number of scenes used), the monthly map and
   the built-up probability (black = 0, yellow = 1). Look for what changes through the year: crops and grass with
   the seasons (real seasonal change, not land use), water with floods, built-up should be stable.
3. **Animation** of the monthly maps.
4. **Year majority** (the most frequent class per pixel), the **built-up change map** (magenta new, cyan lost, dark
   grey built-up throughout) and the **class % per month**.
5. **Table**: our built-up change next to the two references.

The places include hard cases on purpose (Mekong rice fields, Swiss Alps, Sahara dunes, Niamey villages): a demo
that only shows easy places would hide where the method fails.